In [1]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw



Rout = 2
domain = Circle((0, 0), Rout).Face()
domain.edges.name = 'outer'
domain.faces.name = 'outer'


RScat = 0.3


R=1.
inner = Circle((0, 0), R).Face()
inner.edges.name = 'interface'
inner.faces.name = 'inner'


if RScat == 0:
    Draw(Glue([domain-inner,inner]))
    geo = OCCGeometry(Glue([domain-inner,inner]), dim=2)

else:
    scatterer = Circle((0,0),RScat).Face()
    scatterer.edges.name = 'scatterer'


if RScat<R and RScat >0:
    Draw(Glue([domain-inner,inner-scatterer]))
    geo = OCCGeometry(Glue([domain-inner,inner-scatterer]), dim=2)
else:
    Draw(domain-scatterer)
    geo = OCCGeometry(domain-scatterer, dim=2)



WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3…

In [2]:
mesh = Mesh(geo.GenerateMesh(maxh=0.1))
order = 2
mesh.Curve(order)
Draw(mesh)
print(mesh.GetMaterials())
print(mesh.GetBoundaries())

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…

('outer', 'inner')
('outer', 'interface', 'scatterer')


In [3]:

vx = CF((x,y))
rho = sqrt(x**2+y**2)
rho_ = rho-R

# using \omega = R/r^(1/2)

r = rho_/(1-rho_)+1
r_ = 1/(1-rho_)**2
H = 1-1/r_    #characteristic preserving


Pi = OuterProduct(vx,vx)/rho**2
Pi_perp = Id(2)-Pi

#a = R**2*(1-H**2)*r_/rho
a_p = R**2*(1+H)/rho     #characteristic preserving
a_v = (r_*rho/r**2*Pi+1/rho/r_*Pi_perp)

b = R**2*H/rho**2*vx

n = specialcf.normal(2)

In [4]:
fes_p = H1(mesh,order=order)
fes_v = VectorL2(mesh,order=order)

fes_big = fes_p*fes_v

(p,v),(p_,v_) = fes_big.TnT()

M_ = (
    p*p_*dx('inner')
    +v*v_*dx('inner')
    +a_p*p*p_*dx('outer')
    +InnerProduct(a_v*v,v_)*dx('outer')
    )

A_ = (
    grad(p)*v_*dx('inner')-v*grad(p_)*dx('inner')
    -1/r*v*grad(p_)*dx('outer')+r_/r**2/rho/2*InnerProduct(vx,v)*p_*dx('outer')
    +1/r*v_*grad(p)*dx('outer')-r_/r**2/rho/2*InnerProduct(vx,v_)*p*dx('outer')
    -InnerProduct(grad(p),b)*p_*dx('outer')
    +InnerProduct(grad(p_),b)*p*dx('outer')
    -InnerProduct(b,n)*p_*p*ds('outer')
    )

M = BilinearForm(M_).Assemble().mat
A = BilinearForm(A_).Assemble().mat


dt = 1e-2
# matrix for Crank Nicolson time-stepping
Sinv = BilinearForm(M_-dt/2*A_).Assemble().mat.Inverse()

In [5]:
v0 = CF((0,0))
p0 = exp(-30*((x-0.4)**2+(y-0.)**2))


gf = GridFunction(fes_big)

gfp,gfv = gf.components
scenep = Draw(gfp,mesh,min = 0, max = 0.1)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…

In [8]:
gfv.Set(v0)
gfp.Set(p0)

drawevery = (.2/dt)
i = 0

endt = 3
t = 0


gfanim = GridFunction(fes_p,multidim = 0)

while t < endt:
    t += dt
    i += 1
    gf.vec.data += dt*(Sinv@A)*gf.vec
    if (i%drawevery) == 0:
        print('\r t = {}'.format(t),end="")
        #scenep.Redraw()
        gfanim.AddMultiDimComponent(gfp.vec)

 t = 2.99999999999998434

In [9]:
sceneanim = Draw(gfanim,animate=True,min=0,max=0.1)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…